# Interactive Exploration with jupyter-scatter

TMAP integrates with [jupyter-scatter](https://jscatter.dev) for interactive notebook widgets. This notebook covers:

- **Coloring** by continuous or categorical properties
- **Tooltips** with histograms
- **Selection** (lasso, programmatic) and zoom
- **Filtering** to subsets
- **Export** to DataFrame

## Setup

In [ ]:
import pandas as pd

from tmap import TMAP
from tmap.utils import fingerprints_from_smiles, molecular_properties

df = pd.read_csv("../examples/cluster_65053.csv", nrows=2000)
smiles = df["smiles"].tolist()

fps = fingerprints_from_smiles(smiles, fp_type="morgan", radius=2, n_bits=2048)
props = molecular_properties(smiles, properties=["mw", "logp", "n_rings"])

model = TMAP(metric="jaccard", n_neighbors=20, kc=50, seed=42).fit(fps)

# Build a metadata DataFrame for coloring and tooltips
df = pd.DataFrame({"smiles": smiles, **{k: v.tolist() for k, v in props.items()}})


  [Props] 2,000 done, 0 invalid


## 1. Quick Plot with `model.plot()`

The fastest way to get an interactive widget. Pass a DataFrame via `data=` to enable coloring and tooltips.

In [ ]:
scatter = model.plot(
    color_by="mw",
    data=df,
    tooltip_properties=["smiles", "logp", "n_rings"],
    show=False,
    controls=False,
)
scatter.show()


## 2. TmapViz Widget with Controls

`to_tmapviz()` builds a visualization object with multiple color layers. The `to_widget()` method renders it as a notebook widget with a dropdown to switch between layers.

In [ ]:
viz = model.to_tmapviz()
viz.title = "Molecule Explorer"
viz.add_smiles(smiles)
viz.add_color_layout("MW", props["mw"].tolist(), color="viridis")
viz.add_color_layout("LogP", props["logp"].tolist(), color="plasma")
viz.add_color_layout("Ring Count", props["n_rings"].tolist(), categorical=True, color="tab10")

widget = viz.to_widget(width=1000, height=650, controls=True)
widget.show()


/var/folders/zw/c0mr_7jx7t3bz1k6_9n9wh3r0000gn/T/ipykernel_98740/117119249.py:8: UserWarning: Edges are not supported in notebook mode yet and will be ignored.
  widget = viz.to_widget(width=1000, height=650, controls=True)


## 3. Color, Size, and Opacity

`model.plot()` returns a standard `jscatter.Scatter` object, so the full [jscatter API](https://jscatter.dev) is available. You can map properties to size, adjust opacity, toggle axes and legends.

In [ ]:
scatter = model.plot(
    color_by="logp",
    color_map="coolwarm",
    data=df,
    point_size=4,
    opacity=0.9,
    show=False,
)
scatter.size(by="mw", map=[2, 8])
scatter.axes(True)
scatter.legend(True)
scatter.show()


## 4. Tooltips

Enable tooltips to inspect individual points on hover. Add `histograms=True` to see property distributions.

In [11]:
scatter = model.plot(color_by="n_rings", data=df, show=False)
scatter.tooltip(
    enable=True,
    properties=["smiles", "mw", "logp", "n_rings"],
    histograms=True,
    histograms_bins=20,
)
scatter.show()


## 5. Selection and Zoom

Use the lasso tool to select points interactively, or select them programmatically. `scatter.selection()` returns the indices of selected points. `scatter.zoom()` can animate to a selection.

In [12]:
# Lasso-select points interactively, then read them:
# scatter.selection()  -> array of selected indices

# Programmatic selection:
scatter.selection(list(range(50)))
print(f"Selected {len(scatter.selection())} points")

# Zoom to selection
scatter.zoom(to=scatter.selection(), animation=True, padding=0.2)


Selected 50 points


## 6. Filtering

Filter the scatter plot to show only a subset of points. Pass a list of indices to `scatter.filter()`, or `None` to reset.

In [14]:
# Show only molecules with >= 3 rings
heavy = df[df["n_rings"] >= 3].index.tolist()
scatter.filter(heavy)
print(f"Showing {len(heavy)} of {len(df)} points")

# Reset filter
# scatter.filter(None)


Showing 2000 of 2000 points


## 7. Export to DataFrame

Two export paths: `viz.to_dataframe()` for full metadata with coordinates, or use `scatter.selection()` indices to slice your own DataFrame.

In [ ]:
widget = viz.to_widget()
widget.show()


/var/folders/zw/c0mr_7jx7t3bz1k6_9n9wh3r0000gn/T/ipykernel_98740/153501217.py:1: UserWarning: Edges are not supported in notebook mode yet and will be ignored.
  widget = viz.to_widget()


## Select with your mouse
Try holding left click, and then dragging to select multiple points that you can export later!

![Interactive HTML features](../docs/images/selection.gif)

In [22]:
# # Export all metadata + coordinates from TmapViz
# viz_df = viz.to_dataframe(include_coords=True)
# print(f"Exported DataFrame: {viz_df.shape}")
# print(viz_df.head(3))

# Check the points you just selected
print("Index for selected poitns: ",widget.selection())

# Export just selected points
selected_idx = scatter.selection()
if len(selected_idx) > 0:
    selected_df = df.iloc[selected_idx]
    print(f"\nSelected subset: {selected_df.shape}")
    print(selected_df.head(3))


Index for selected poitns:  [1074 1753 1752 1748 1749 1755 1715 1516  452  458  454 1939  459 1179
 1654  457 1655  393 1681 1940 1649 1641 1565 1890  364 1891 1762 1772
 1714 1763  394 1773  496  382  495  425 1909 1910 1515  383 1636 1825
 1137 1142 1138 1771 1774 1139 1624 1201  583  436 1141 1161  163 1160
 1165  437  557  453  430  483 1140  482 1399  811  559 1162  426  464
 1166 1167 1131  456 1594  392 1756 1750 1888 1635 1889]

Selected subset: (63, 4)
                                                 smiles         mw    logp  \
108       COC1=CC=C(Br)C(S(=O)(=O)N2CCCCCC2C2CCCO2)=C1F  435.05152  3.7091   
1299  COC1=CC=C(F)C(S(=O)(=O)N2CCN(C3CCC(C)CC3)CC2)=C1F  388.16322  2.8584   
1300  COC1=CC=C(F)C(S(=O)(=O)N2CCN3C4=CC=CC=C4CCC3C2...  394.11627  2.7991   

      n_rings  
108       3.0  
1299      3.0  
1300      4.0  


## Notes

- `model.plot()` is the quickest path -- returns a `jscatter.Scatter` directly
- `viz.to_widget()` adds a color-switching dropdown for multiple layers
- Tree edges are not drawn in notebook widgets -- use `model.to_html()` for edge rendering
- Full jscatter API: https://jscatter.dev